# EPL Result Modeling: Elo Baseline and Stacked XGBoost

This notebook uses `epl_model_features.csv` to:

1. fit an Elo-only multinomial logistic regression baseline
2. generate held-out Elo probabilities for `2024/25`
3. train a stacked XGBoost model using `EloProb_*` plus pre-match recent-form features
4. compare multiclass log loss and accuracy

The split is time-based and not shuffled:

- Train: `2020/21` through `2023/24`
- Validation/Test: `2024/25`

For the stacked model, the validation Elo probabilities come from a logistic regression trained only on the training split. The XGBoost training rows use expanding-window out-of-fold Elo probabilities by season so the stack stays time-aware.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

RESULT_ORDER = ["H", "D", "A"]
RESULT_TO_INT = {result: idx for idx, result in enumerate(RESULT_ORDER)}
INT_TO_RESULT = {idx: result for result, idx in RESULT_TO_INT.items()}

TRAIN_SEASONS = ["2020/21", "2021/22", "2022/23", "2023/24"]
VALID_SEASON = "2024/25"

project_root = Path.cwd().resolve()
if not (project_root / "data").exists():
    project_root = project_root.parent

DATA_PATH = project_root / "data" / "epl_model_features.csv"
DATA_PATH

In [ ]:
df = pd.read_csv(DATA_PATH, na_values=["NaN"]).copy()
df["MatchDate"] = pd.to_datetime(df["MatchDate"])

numeric_columns = [
    "EloDiff",
    "Home_GoalsScored_Last5",
    "Away_GoalsScored_Last5",
    "GoalsConceded_Last5_Diff",
    "Shots_Last5_Diff",
    "SOT_Last5_Diff",
    "Home_ShotsAllowed_Last5",
    "Away_ShotsAllowed_Last5",
    "Home_SOTAllowed_Last5",
    "Away_SOTAllowed_Last5",
    "PPG_Last5_Diff",
    "RestDiff",
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["GoalsScored_Last5_Diff"] = (
    df["Home_GoalsScored_Last5"] - df["Away_GoalsScored_Last5"]
)
df["ShotsAllowed_Last5_Diff"] = (
    df["Home_ShotsAllowed_Last5"] - df["Away_ShotsAllowed_Last5"]
)
df["SOTAllowed_Last5_Diff"] = (
    df["Home_SOTAllowed_Last5"] - df["Away_SOTAllowed_Last5"]
)
df["target"] = df["FullTimeResult"].map(RESULT_TO_INT)

display(df[["Season", "MatchDate", "HomeTeam", "AwayTeam", "EloDiff", "PPG_Last5_Diff", "GoalsScored_Last5_Diff", "ShotsAllowed_Last5_Diff", "RestDiff", "FullTimeResult"]].head())
display(df["Season"].value_counts(sort=False).rename("Matches"))

## Step 1: Elo Baseline Regression

Baseline model:

- input: `EloDiff`
- target: `FullTimeResult` with classes `H`, `D`, `A`
- model: multinomial logistic regression

This produces smooth baseline probabilities:

- `EloProb_H`
- `EloProb_D`
- `EloProb_A`

In [ ]:
train_df = df.loc[df["Season"].isin(TRAIN_SEASONS)].copy()
valid_df = df.loc[df["Season"] == VALID_SEASON].copy()

baseline_features = ["EloDiff"]

baseline_model = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    random_state=42,
)
baseline_model.fit(train_df[baseline_features], train_df["target"])

valid_baseline_probs = baseline_model.predict_proba(valid_df[baseline_features])
valid_df[["EloProb_H", "EloProb_D", "EloProb_A"]] = valid_baseline_probs
valid_df["EloPred"] = valid_baseline_probs.argmax(axis=1)
valid_df["EloPredLabel"] = valid_df["EloPred"].map(INT_TO_RESULT)

baseline_metrics = pd.DataFrame(
    [
        {
            "Model": "Logistic Regression (Elo only)",
            "LogLoss": log_loss(valid_df["target"], valid_baseline_probs, labels=[0, 1, 2]),
            "Accuracy": accuracy_score(valid_df["target"], valid_df["EloPred"]),
        }
    ]
)

display(baseline_metrics)
display(valid_df[["Season", "MatchDate", "HomeTeam", "AwayTeam", "EloDiff", "EloProb_H", "EloProb_D", "EloProb_A", "FullTimeResult", "EloPredLabel"]].head())

## Step 2: XGBoost With Extra Features

Stacked setup:

- baseline Elo model provides `EloProb_H`, `EloProb_D`, `EloProb_A`
- XGBoost consumes those probabilities plus recent-game pre-match features

To keep the stack honest, the training rows get season-based expanding-window out-of-fold Elo probabilities. That means the first training season (`2020/21`) has no prior-season baseline model available and is excluded from the stacked XGBoost training fit.

In [ ]:
def make_probability_frame(probabilities, index, prefix):
    columns = [f"{prefix}_H", f"{prefix}_D", f"{prefix}_A"]
    return pd.DataFrame(probabilities, index=index, columns=columns)


def expanding_window_elo_oof(train_frame, seasons, feature_name="EloDiff"):
    oof_frames = []

    for season_idx in range(1, len(seasons)):
        fit_seasons = seasons[:season_idx]
        score_season = seasons[season_idx]

        fit_frame = train_frame.loc[train_frame["Season"].isin(fit_seasons)]
        score_frame = train_frame.loc[train_frame["Season"] == score_season].copy()

        model = LogisticRegression(
            solver="lbfgs",
            max_iter=1000,
            random_state=42,
        )
        model.fit(fit_frame[[feature_name]], fit_frame["target"])
        score_probs = model.predict_proba(score_frame[[feature_name]])
        score_frame[["EloProb_H", "EloProb_D", "EloProb_A"]] = score_probs
        oof_frames.append(score_frame)

    stacked_train = pd.concat(oof_frames).sort_index()
    return stacked_train


stack_train_df = expanding_window_elo_oof(train_df, TRAIN_SEASONS)
stack_valid_df = valid_df.copy()

stack_feature_columns = [
    "EloProb_H",
    "EloProb_D",
    "EloProb_A",
    "PPG_Last5_Diff",
    "GoalsScored_Last5_Diff",
    "GoalsConceded_Last5_Diff",
    "Shots_Last5_Diff",
    "SOT_Last5_Diff",
    "ShotsAllowed_Last5_Diff",
    "SOTAllowed_Last5_Diff",
    "RestDiff",
]

xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    min_child_weight=1,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=42,
)

xgb_model.fit(stack_train_df[stack_feature_columns], stack_train_df["target"])

valid_xgb_probs = xgb_model.predict_proba(stack_valid_df[stack_feature_columns])
stack_valid_df[["XGBProb_H", "XGBProb_D", "XGBProb_A"]] = valid_xgb_probs
stack_valid_df["XGBPred"] = valid_xgb_probs.argmax(axis=1)
stack_valid_df["XGBPredLabel"] = stack_valid_df["XGBPred"].map(INT_TO_RESULT)

xgb_metrics = pd.DataFrame(
    [
        {
            "Model": "XGBoost (stacked on Elo probabilities)",
            "LogLoss": log_loss(stack_valid_df["target"], valid_xgb_probs, labels=[0, 1, 2]),
            "Accuracy": accuracy_score(stack_valid_df["target"], stack_valid_df["XGBPred"]),
        }
    ]
)

print(f"Stacked XGBoost training rows: {len(stack_train_df):,}")
print(f"Rows dropped from stacked training due to no prior-season Elo baseline: {len(train_df) - len(stack_train_df):,}")

In [ ]:
comparison = pd.concat([baseline_metrics, xgb_metrics], ignore_index=True)
display(comparison)

validation_scored = stack_valid_df[
    [
        "Season",
        "MatchDate",
        "HomeTeam",
        "AwayTeam",
        "FullTimeResult",
        "EloDiff",
        "EloProb_H",
        "EloProb_D",
        "EloProb_A",
        "XGBProb_H",
        "XGBProb_D",
        "XGBProb_A",
        "EloPredLabel",
        "XGBPredLabel",
    ]
].copy()

display(validation_scored.head(10))

validation_scored.describe(include="all")